In [1]:
from PIL import Image
import import_ipynb
def Brezenheim_otrezok(img, x0, y0, x1, y1):
    b = int((img.size[0] - 1) / 2)
    pixels = img.load()
    steep = abs(y1 - y0) > abs(x1 - x0)
    if steep:
        x0, y0 = y0, x0
        x1, y1 = y1, x1
    if x0 > x1:
        x0, x1 = x1, x0
        y0, y1 = y1, y0
    dx = x1 - x0
    dy = abs(y1 - y0)
    d = 2 * dy - dx
    y = y0
    step_y = 1 if y1 > y0 else -1
    for x in range(x0, x1 + 1):
        curr_x, curr_y = (y, x) if steep else (x, y)
        pixels[b + curr_x, b - curr_y] = 0
        if d > 0:
            y += step_y
            d -= 2 * dx
        d += 2 * dy

На основе реализованного алгоритма Брезенхема для рисования отрезка построить «проволочный» рендер заданной 3D модели (можно использовать свою модель) с использованием файла описания ее геометрии obj.

In [4]:
def render(file):
    vertices = []
    faces = []
    with open(file, 'r') as f:
        for line in f:
            if line.startswith('v '):
                vertices.append(list(map(float, line.split()[1:4])))
            elif line.startswith('f '):
                faces.append([int(i.split('/')[0]) - 1 for i in line.split()[1:]])
    img = Image.new('L', (501, 501), color=255)
    def project(vertex):
        x = int((vertex[0] ) * 150)
        y = int(vertex[1] * 150)
        return x, y
    for face in faces:
        for i in range(len(face)):
            v1 = vertices[face[i]]
            v2 = vertices[face[(i + 1) % len(face)]]
            x0, y0 = project(v1)
            x1, y1 = project(v2)
            Brezenheim_otrezok(img, x0, y0, x1, y1)
    img.show()


In [5]:
render('african_head.obj')